# 오늘춥나?? — 1단계: 학습 데이터 합치기

안양만안(377) AWS 관측값과 안양대 500m 격자 값을 **같은 시각끼리 한 줄로** 붙여요. 파일은 모두 구글 드라이브에서 읽어요.

**준비물 (구글 드라이브 `내 드라이브/onulchupna/` 안)**
- `aws/` 폴더: 친구가 보내준 `aws_377.csv` (또는 기상자료개방포털에서 받은 AWS CSV 원본들). 직접 이 폴더에 올려주세요.
- `grid/` 폴더: 격자 노트북(`anyang_grid_colab.ipynb`)이 자동 저장한 `grid_YYYY_MM.csv`들

**결과:** `onulchupna/train_data.csv` 하나 (2단계 학습에서 이 파일만 써요)

**순서:** ① 드라이브 연결 → ② AWS 읽기 → ③ 격자 읽기 → ④ 합치기 → ⑤ 확인 → ⑥ 저장

## ① 드라이브 연결

In [ ]:
import os, glob
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/onulchupna"
AWS_FILES = sorted(glob.glob(f"{BASE}/aws/*.csv"))
GRID_FILES = sorted(glob.glob(f"{BASE}/grid/grid_*.csv"))
print("AWS 파일", len(AWS_FILES), "개:", [os.path.basename(f) for f in AWS_FILES])
print("격자 파일", len(GRID_FILES), "개")

## ② AWS 읽기

정리된 `aws_377.csv`든, 기상청 원본 CSV든 둘 다 읽을 수 있어요.

In [ ]:
def read_kma_csv(path):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"파일을 읽을 수 없어요: {path}")

def clean_aws(raw):
    """기상청 AWS CSV → tm, aws_ta, aws_hm, aws_ws (이미 정리된 파일이면 그대로)"""
    raw.columns = [str(c).strip() for c in raw.columns]
    if {"tm", "aws_ta", "aws_hm", "aws_ws"} <= set(raw.columns):
        df = raw[["tm", "aws_ta", "aws_hm", "aws_ws"]].copy()
        df["tm"] = pd.to_datetime(df["tm"])
        return df
    def col(prefix):
        hit = [c for c in raw.columns if c.startswith(prefix)]
        if not hit:
            raise KeyError(f"'{prefix}' 열이 없어요. 이 파일의 열: {list(raw.columns)}")
        return hit[0]
    return pd.DataFrame({
        "tm":     pd.to_datetime(raw[col("일시")]),
        "aws_ta": pd.to_numeric(raw[col("기온")], errors="coerce"),
        "aws_hm": pd.to_numeric(raw[col("습도")], errors="coerce"),
        "aws_ws": pd.to_numeric(raw[col("풍속")], errors="coerce"),
    })

aws = pd.concat([clean_aws(read_kma_csv(f)) for f in AWS_FILES], ignore_index=True)
aws = aws.drop_duplicates("tm").sort_values("tm")
print(len(aws), "줄,", aws["tm"].min(), "~", aws["tm"].max())
aws.head()

## ③ 격자 읽기

In [ ]:
g = pd.concat([pd.read_csv(f, encoding="utf-8-sig") for f in GRID_FILES], ignore_index=True)
grid = pd.DataFrame({
    "tm":     pd.to_datetime(g["tm"]),
    "uni_ta": pd.to_numeric(g["ta"], errors="coerce"),
    "uni_hm": pd.to_numeric(g["hm"], errors="coerce"),
    "uni_ws": pd.to_numeric(g["ws_10m"], errors="coerce"),
}).drop_duplicates("tm").sort_values("tm")
print(len(grid), "줄,", grid["tm"].min(), "~", grid["tm"].max())
grid.head()

## ④ 합치기

같은 시각(정시)끼리 붙이고, 결측과 말이 안 되는 값을 버려요.
- 습도 0~100%, 풍속 0~60 m/s, 기온 -40~50°C 밖 → 버림 (AWS 파일에 품질 플래그 열이 없어서 범위 검사로 대신해요)

In [ ]:
for df in (aws, grid):
    df["tm"] = df["tm"].dt.floor("h")

data = pd.merge(aws, grid, on="tm", how="inner")

def valid(df, p):
    return (df[f"{p}_ta"].between(-40, 50) & df[f"{p}_hm"].between(0, 100) & df[f"{p}_ws"].between(0, 60))

before = len(data)
data = data[valid(data, "aws") & valid(data, "uni")].reset_index(drop=True)
data["month"] = data["tm"].dt.month
data["hour"] = data["tm"].dt.hour
print(f"합친 줄 {before} → 정상 값만 {len(data)} ({before - len(data)}줄 버림)")
data.head()

## ⑤ 확인: 안양대와 만안의 평균 차이

In [ ]:
print("기온 차이 (안양대 - 만안) 평균:", round((data.uni_ta - data.aws_ta).mean(), 2), "°C")
print("습도 차이 (안양대 - 만안) 평균:", round((data.uni_hm - data.aws_hm).mean(), 1), "%")
calm = data.aws_ws >= 0.5   # 바람이 거의 없을 때는 비율이 튀어서 제외
print("풍속 비율 (안양대 / 만안) 중앙값:", round((data.uni_ws[calm] / data.aws_ws[calm]).median(), 2))

(data.assign(diff=data.uni_ta - data.aws_ta)
     .groupby("hour")["diff"].mean().round(2).to_frame("기온 차이(°C)").T)

## ⑥ 드라이브에 저장

In [ ]:
out = f"{BASE}/train_data.csv"
data.to_csv(out, index=False, encoding="utf-8-sig")
print("저장:", out, len(data), "줄")